# Human-in-the-Loop: Pre-Action Gates, Risk Tiering, and Audit Logging

The README for this lesson introduces Human-in-the-Loop with a short snippet that asks the user `APPROVE` or `REJECT` after the agent has already produced a response. That pattern is a fine starting point, but production HITL implementations commonly need three additional pieces:

1. A **pre-action gate** that runs **before** the agent executes a risky step, so cost, irreversibility, and latency stay under control.
2. **Risk tiering**, so low-risk actions auto-execute, medium-risk actions are batch-approved, and only high-risk actions block on a human.
3. An **audit log plus revision loop**, so every gate decision is recorded as JSONL, and a rejection re-prompts the agent with a structured reason instead of just printing `Revising...`.

This notebook builds each of these on top of the same primitives as `06-system-message-framework.ipynb`. It runs end-to-end in `DEMO_MODE = True` (no interactive input needed) or with real `input()` prompts when `DEMO_MODE = False`. Note: in DEMO_MODE the third goal's retry is scripted so the loop mechanics are visible end-to-end. Real revision-driven re-classification requires `DEMO_MODE = False` and an operator.

**Out of scope (handled in other lessons):** authentication and access control (Lesson 06 README threat 2), tool-call middleware (Lesson 14 MAF deep dive), multi-agent debate patterns.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Uzorak 1: Prethodni čuvar

README-ov HITL isječak najprije poziva agenta, a zatim traži od korisnika da odobri izlaz. To je **post-akcijski** tok. Agent je već izvršio radnju, pa je LLM poziv već plaćen, i bilo koji nuspojava (poslana e-pošta, upisan redak u bazu podataka, objavljen komentar) se već dogodila.

**Pre-akcijski** tok umeće bravu prije nego agent pokrene rizični korak. Agent predlaže radnju, brava odlučuje hoće li se izvršiti, i nuspojava nastaje samo uz odobrenje.

| Aspekt | Post-akcijsko odobrenje (README isječak) | Pre-akcijska brava (ovaj bilježnik) |
|---|---|---|
| Kada se izvršava odobrenje? | Nakon što je agent proizveo izlaz | Prije bilo kakvog izvršenja nuspojave |
| Trošak LLM-a pri odbijanju | Već plaćen | Plaćen samo za prijedlog, ne za radnju |
| Nepovratne nuspojave | Moguće (radnja se već dogodila) | Spriječeno |
| Jasnoća revizije | Odobrenje je ispisna naredba | Odobrenje je JSON zapis s vremenom, radnjom i razlogom |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Uzorak 2: Rangiranje rizika

Nije svaka radnja potrebna ljudska odobrenja. Pregled samo za čitanje putem javnog API-ja ima drugačiji značaj od slanja e-pošte kupcu. Postupanje s oboje jednako troši pozornost operatera i usporava agenta.

Jednostavan model od 3 razine:

| Razina | Primjeri | Tijek odobrenja |
|---|---|---|
| `nisko` (samo za čitanje) | Pretraživanje baze znanja, pregled opcija leta, dohvaćanje javne web stranice | Automatsko izvršenje, evidentirano za reviziju |
| `srednje` (jeftina mutacija) | Predmemoriranje rezultata, inkrement brojača, zakazivanje podsjetnika | Automatsko izvršenje, ali pregled u dnevnim skupinama |
| `visoko` (usmjereno prema van ili nepovratno) | Slanje e-pošte, naplata kartice, objavljivanje na javnom kanalu | Blokira se do ljudskog odobrenja |

Ovo je jedno rangiranje. Produkcijski sustavi često koriste detaljnije razine (npr. razine dozvola AWS IAM-a, razine pristupa temeljene na ulogama). Verzija s 3 razine u nastavku je najmanja korisna verzija za agenta koji kombinira radnje samo za čitanje i one sa sporednim učincima.

Klasifikator u nastavku koristi heuristiku ključnih riječi kako bi demo ostao determinističan i jeftin. U produkcijskom sustavu zamijenili biste ga naučenim klasifikatorom ili sustavom politika.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Uzorak 3: Dnevnik revizije i petlja revizije

`print("Response approved.")` nije dnevnik revizije. Za povjerenje, svaka odluka vrata trebala bi se zabilježiti kao strukturirani događaj koji kasnije možete pretraživati, reproducirati ili priložiti za pregled incidenta.

Dvije stvari:

1. **Samo dodavanje JSONL.** Jedan redak po odluci, sa vremenskom oznakom, akcijom, razinom, odlukom, razlogom. Jednostavno za grep, jednostavno za kasnije slanje u stvarni spremnik dnevnika.
2. **Petlja revizije kod odbijanja.** Kad vrata vrate `deny`, agent ponovno daje upit samom sebi s razlogom odbijanja u kontekstu, kako bi sljedeći prijedlog mogao izbjeći problem.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Dodatni izvori

Više drugih javnih projekata implementira varijacije ovih HITL obrazaca. Usporedite pristupe kako biste pronašli što odgovara vašem stacku:

- **LangChain** alati s ljudskim uplitanjem ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): alati koji se mogu jednostavno dodati, a koji pauziraju izvođenje za unos čovjeka.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ je ovo restrukturirao): koristi agentovu ulogu specifično za predstavljanje čovjeka u multi-agentnim razgovorima.
- **Semantic Kernel** funkcijski filtri ([docs](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): middleware koji se izvršava oko svakog poziva funkcije, prikladan za kontrolu logike.

Svaki projekt različito pristupa trima pod-obrascima: LangChain ih omotava kao alate, AutoGen koristi ulogu agenta, Semantic Kernel koristi middleware filtre. Pročitajte jednu ili dvije implementacije u cijelosti prije nego što odaberete dizajn za svog vlastitog agenta.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
